# 03 – Character Nicknames

Esplorazione e data cleaning del dataset `character_nicknames.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `character_mal_id` | Chiave estern che fa riferimento alla chiave primaria di ⁠ characters.csv |
| `nickname` | Soprannome del personaggio |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [1]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_nicknames = pd.read_csv('../datasets/character_nicknames.csv')
print(f'Shape: {df_nicknames.shape}')
df_nicknames.info()
df_nicknames.head()

Shape: (37080, 2)
<class 'pandas.DataFrame'>
RangeIndex: 37080 entries, 0 to 37079
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   character_mal_id  37080 non-null  int64
 1   nickname          37064 non-null  str  
dtypes: int64(1), str(1)
memory usage: 579.5 KB


,character_mal_id,nickname
0,280205,Hikaruko
1,280129,Hinacchi
2,280127,Bertha Willis
3,280066,Jimmy
4,280059,Full Body Red Square


In [3]:
n_originale = len(df_nicknames)

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [4]:
mask_dup = df_nicknames.duplicated(keep=False)       # tutte le occorrenze coinvolte
n_righe_coinvolte = mask_dup.sum()                    # righe totali che hanno almeno un duplicato
n_gruppi = df_nicknames[mask_dup].duplicated(keep='first').sum()  # occorrenze extra (da rimuovere)
n_tenute = n_righe_coinvolte - n_gruppi               # prime occorrenze mantenute

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_nicknames.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_nicknames):,}')

Righe totali coinvolte in duplicazioni : 168
  → prime occorrenze mantenute         : 66
  → occorrenze extra rimosse           : 102

Righe prima della rimozione : 37,080
Righe dopo la rimozione     : 36,978


## 2. Analisi colonna per colonna

### 2.1 `character_mal_id`

In [5]:
analyze(df_nicknames['character_mal_id'])


════════════════════════════════════════════════════════════════════════════════
═════════════════  🔬  SERIES ANALYSIS  —  'character_mal_id'  ══════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    character_mal_id
  dtype                          int64
  Shape                          36,978
  Index range                    0 … 37079
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     36,978
  Non-null values                36,978  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Zeros                          0  (0.00%)
  Positives                      36,978   (100.00%)
  Negatives                      0   (0.00%)
  Unique values                  30,282  (81.89%)

 📐 Descriptive Statistics
─────────────────────────────────────────────────────────────

**Nessuna pulizia necessaria.**  
Nessun null, dtype già `int64`. I duplicati sono attesi: uno stesso personaggio può avere più nickname.

In [6]:
# Nessuna operazione richiesta
print(f'Null in character_mal_id  : {df_nicknames["character_mal_id"].isna().sum()}')
print(f'Duplicati in character_mal_id (attesi) : {df_nicknames["character_mal_id"].duplicated().sum():,}')

Null in character_mal_id  : 0
Duplicati in character_mal_id (attesi) : 6,696


### 2.2 `nickname`

In [7]:
analyze(df_nicknames['nickname'])


════════════════════════════════════════════════════════════════════════════════
═════════════════════  🔬  SERIES ANALYSIS  —  'nickname'  ══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    nickname
  dtype                          str
  Shape                          36,978
  Index range                    0 … 37079
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     36,978
  Non-null values                36,962  [██████████████████████████████] 100.0%
  Null / NaN                     16  (0.04%)
  Empty strings                  0
  Unique values                  28,928  (78.23%)
  Duplicate values               8,034

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     1  chars
  Max 

**Pulizie necessarie:**
- 16 null residui → rimuovere le righe (una riga senza nickname non ha significato)
- I nickname duplicati sono attesi: soprannomi comuni (es. "King") possono comparire per personaggi diversi

In [8]:
print(f'Null in nickname prima della pulizia: {df_nicknames["nickname"].isna().sum()}')

df_nicknames.dropna(subset=['nickname'], inplace=True)

print(f'Null in nickname dopo la pulizia    : {df_nicknames["nickname"].isna().sum()}')
print(f'Righe dopo pulizia nickname         : {len(df_nicknames):,}')

Null in nickname prima della pulizia: 16
Null in nickname dopo la pulizia    : 0
Righe dopo pulizia nickname         : 36,962


### 2.3 Chiave primaria `(character_mal_id, nickname)`

La chiave primaria naturale è la coppia `(character_mal_id, nickname)`: verifichiamo che sia univoca dopo la pulizia.

In [9]:
n_pk_dup = df_nicknames.duplicated(subset=['character_mal_id', 'nickname'], keep=False).sum()
print(f'Duplicati su (character_mal_id, nickname): {n_pk_dup}')

if n_pk_dup > 0:
    print('→ Rimozione duplicati sulla chiave primaria...')
    df_nicknames.drop_duplicates(subset=['character_mal_id', 'nickname'], keep='first', inplace=True)
    print(f'Righe dopo la rimozione: {len(df_nicknames):,}')
else:
    print('→ Chiave primaria già univoca, nessuna operazione richiesta.')

Duplicati su (character_mal_id, nickname): 0
→ Chiave primaria già univoca, nessuna operazione richiesta.


## 3. Data Cleaning

In [10]:
print('=== Riepilogo Dataset Pulito ===')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_nicknames):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_nicknames):>10,}')
print()
print('Dtypes finali:')
print(df_nicknames.dtypes)
print()
print('Valori mancanti residui:')
print(df_nicknames.isnull().sum())

=== Riepilogo Dataset Pulito ===
Righe originali      :     37,080
Righe dopo cleaning  :     36,962
Righe rimosse totali :        118

Dtypes finali:
character_mal_id    int64
nickname              str
dtype: object

Valori mancanti residui:
character_mal_id    0
nickname            0
dtype: int64


In [11]:
df_nicknames.head()

,character_mal_id,nickname
0,280205,Hikaruko
1,280129,Hinacchi
2,280127,Bertha Willis
3,280066,Jimmy
4,280059,Full Body Red Square


## 4. Salvataggio dataset pulito

In [12]:
df_nicknames.to_csv('../datasets_cleaned/character_nicknames_clean.csv', index=False)
print('Salvato: datasets_cleaned/character_nicknames_clean.csv')

Salvato: datasets_cleaned/character_nicknames_clean.csv
